# 🎮 Epic Games Store - Data Preprocessing for Prediction Model

**Objective:** Load, clean, and preprocess the `Epic.csv` data to prepare it for a machine learning model that predicts the time until a game becomes free on the service.

---

### 📋 **Steps:**
1. Load the `Epic.csv` dataset.
2. Inspect and clean the data (handle missing values, correct data types).
3. Calculate the **`Time_to_Service_Days`** (our primary target variable).
4. Define the categorical **`Prediction_Target`** bins based on `Time_to_Service_Days`.
5. Discuss strategies for handling multi-entry `Publisher`/`Developer` columns.
6. Recommend a suitable machine learning model.
7. Provide a template sentence for explaining predictions.
8. Save the preprocessed data to a new CSV file.

---

### ⚙️ **Prerequisites**
Ensure `Epic.csv` is in the same directory as this notebook. Run the cell below to install necessary libraries.

In [ ]:
# Cell 1: Install Libraries
!pip install pandas numpy

In [ ]:
# Cell 2: Load and Inspect Data

import pandas as pd
import numpy as np
import os

print(f"Current working directory: {os.getcwd()}")
epic_file = 'Epic.csv'

try:
    df = pd.read_csv(epic_file)
    print(f"\n✅ Successfully loaded '{epic_file}'.")
    
    print("\n--- Initial DataFrame Head ---")
    print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
    
    print("\n--- Initial DataFrame Info ---")
    df.info()
    
except FileNotFoundError:
    print(f"❌ ERROR: File '{epic_file}' not found in the current directory.")
    df = None # Set df to None to prevent errors in subsequent cells if loading fails
except Exception as e:
    print(f"❌ ERROR loading file: {e}")
    df = None

# Store original columns for reference
original_columns = list(df.columns) if df is not None else []
print(f"\nOriginal columns: {original_columns}")

### ✨ **Step 3: Data Cleaning and Basic Preprocessing**

Rename columns, handle missing critical values, and convert date strings to datetime objects.

In [ ]:
# Cell 3: Clean Data and Convert Types

if df is not None:
    # Rename columns for consistency 
    # Adjust mapping based on the actual columns identified in df.info()
    column_mapping = {
        original_columns[0]: 'Game', # Assuming first column is game name
        original_columns[1]: 'Release_Date',
        original_columns[2]: 'Service_Date',
        original_columns[3]: 'Removed_from_Service',
        original_columns[4]: 'Metacritic_Score',
        original_columns[5]: 'Publisher',
        original_columns[6]: 'Developer'
    }
    # Ensure all original columns expected are present before renaming
    valid_mapping = {k: v for k, v in column_mapping.items() if k in df.columns}
    df.rename(columns=valid_mapping, inplace=True)
    
    print("\nRenamed columns:", list(df.columns))

    # Drop potential header/junk rows (like the first row in the previous inspection)
    if df.iloc[0].isnull().all() or pd.isna(df.iloc[0]['Game']):
       df = df.iloc[1:].copy()
       print("Dropped first row assuming it was metadata.")

    # Drop rows where essential information is missing
    initial_rows = len(df)
    df.dropna(subset=['Game', 'Service_Date', 'Release_Date'], inplace=True)
    print(f"Dropped {initial_rows - len(df)} rows with missing Game, Service_Date, or Release_Date.")

    # Convert date columns to datetime objects
    df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
    df['Service_Date'] = pd.to_datetime(df['Service_Date'], errors='coerce')

    # Drop rows where date conversion failed
    initial_rows = len(df)
    df.dropna(subset=['Release_Date', 'Service_Date'], inplace=True)
    print(f"Dropped {initial_rows - len(df)} rows with invalid date formats.")

    # Clean Metacritic Score - Convert to nullable integer
    df['Metacritic_Score'] = pd.to_numeric(df['Metacritic_Score'], errors='coerce')
    df['Metacritic_Score'] = df['Metacritic_Score'].round(0).astype('Int64') 

    print("\n--- Cleaned DataFrame Info ---")
    df.info()
else:
    print("DataFrame not loaded. Cannot proceed with cleaning.")

### ⏱️ **Step 4: Calculate Target Variable (`Time_to_Service_Days`)**

Compute the number of days between the game's release and when it was added to the Epic Games Store for free.

In [ ]:
# Cell 4: Calculate Time to Service

if df is not None:
    # Calculate the time difference in days
    df['Time_to_Service_Days'] = (df['Service_Date'] - df['Release_Date']).dt.days

    # Handle cases where service date might be before release date (Day 0/1)
    df['Time_to_Service_Days'] = df['Time_to_Service_Days'].apply(lambda x: max(0, x if pd.notna(x) else -1))
    
    # Remove rows where calculation resulted in invalid number (-1)
    df = df[df['Time_to_Service_Days'] >= 0] 

    print("\n--- Data with Time_to_Service_Days ---")
    print(df[['Game', 'Release_Date', 'Service_Date', 'Time_to_Service_Days']].head().to_markdown(index=False, numalign="left", stralign="left"))
else:
    print("DataFrame not loaded. Cannot calculate Time to Service.")

### 🏷️ **Step 5: Define Categorical Target Bins (`Prediction_Target`)**

Convert the continuous `Time_to_Service_Days` into the discrete categories requested for the prediction model.

In [ ]:
# Cell 5: Create Prediction Target Bins

if df is not None:
    # Define the classification bins based on user request
    def classify_time_to_service(days):
        if pd.isna(days):
            return 'Unknown' 
        elif days <= 180:   # Within ~6 Months
            return '0 - 6 Months'
        elif days <= 545:  # Next 12 months (approx 6 to 18 months from release)
            return '6 - 18 Months' 
        elif days <= 1095: # More than 12 months (approx 18 to 36 months from release)
             return '18 - 36 Months'
        else: # Older than 3 years
            return '36+ Months / Never'

    df['Prediction_Target'] = df['Time_to_Service_Days'].apply(classify_time_to_service)

    print("\n--- Data with Prediction Target Bins ---")
    print(df[['Game', 'Time_to_Service_Days', 'Prediction_Target']].head().to_markdown(index=False, numalign="left", stralign="left"))
    print("\nDistribution of Target Bins:")
    print(df['Prediction_Target'].value_counts().to_markdown())
else:
    print("DataFrame not loaded. Cannot create target bins.")

### 👥 **Step 6: Handling Multi-Entry Publishers/Developers**

The `Publisher` and `Developer` columns sometimes contain multiple entries (e.g., `"Publisher A, Publisher B"`). For modeling, these need special treatment.

**Recommended Strategy (Future Step):**
1. **Combine Data:** After preprocessing `Xbox.csv` and `PS.csv`, merge all three datasets.
2. **Explode:** Temporarily split rows with multiple publishers/developers into separate rows, one for each entity.
3. **Aggregate:** Calculate metrics for *each unique entity* across the entire dataset:
    * `Avg_Time_to_Service`: Average days it takes for games from this entity to hit *any* service.
    * `Total_Inclusion_Count`: How many times games from this entity appeared on *any* service.
4. **Merge Back:** Add these numerical metrics (`Publisher_Avg_TTS`, `Publisher_Count`, `Developer_Avg_TTS`, `Developer_Count`) to the main DataFrame. For games with multiple original entities, you might average their individual metrics.

**For now, we will keep these columns as strings.** The feature engineering step will happen after all data sources are combined.

### 🤖 **Step 7: Model Recommendation**

Based on the prepared data, which includes:
* **Numerical Features:** `Metacritic_Score` (potentially imputed), and the *future* engineered features like `Publisher_Avg_TTS`, `Publisher_Total_Inclusions`, `Developer_Avg_TTS`, `Developer_Total_Inclusions`.
* **Categorical Features:** `Service` (Epic, Xbox, PS - to be added later).
* **Target Variable:** `Prediction_Target` (Categorical bins: '0-6 Months', '6-18 Months', etc.).

A **Classification Model** is appropriate. Good choices include:

1.  **Random Forest Classifier:** Excellent for handling mixed data types, capturing non-linear relationships, and is relatively robust to outliers. It also provides feature importance scores.
2.  **Gradient Boosting Classifier (like XGBoost, LightGBM, CatBoost):** Often achieve higher accuracy than Random Forests but may require more careful tuning. They are powerful and widely used.
3.  **Logistic Regression (with One-Hot Encoding):** A simpler baseline model. Requires careful handling of categorical features (like Publisher/Developer names if not using engineered metrics) via One-Hot Encoding.

**Recommendation:** Start with **Random Forest Classifier** due to its ease of use and good performance on this type of tabular data.

### 📝 **Step 8: Prebaked Sentence Template for Predictions**

As requested, here's a template for explaining the model's prediction. You would fill the placeholders (`[...]`) with the actual input data and results from your trained model.

```text
Based on the game's publisher ([Publisher Name(s)]) and developer ([Developer Name(s)]), which historically have an average time-to-service of approximately [Publisher_Avg_TTS / 30:.1f] months across [Publisher_Sum_Inclusions] total inclusions on tracked services, and considering its Metacritic score of [Metacritic Score], the model predicts the likelihood of this game appearing on [Target Service Name] as:

**Prediction:** [Prediction_Target (e.g., 6 - 18 Months)] 
(Confidence: [Model Confidence Score:.1f]%)
```

**Note:** The `[Model Confidence Score]` would come from the model's `predict_proba` method. The average time and inclusion counts are the features engineered in Step 6.

### 💾 **Step 9: Save Preprocessed Data**

Save the cleaned and processed Epic Games data to a new CSV file for the next stage (combining with Xbox/PS data and feature engineering).

In [ ]:
# Cell 6: Save Processed Data

if df is not None:
    output_filename = 'Epic_Preprocessed.csv'
    # Select columns to save 
    columns_to_save = [
        'Game', 'Release_Date', 'Service_Date', 'Metacritic_Score', 
        'Publisher', 'Developer', 'Time_to_Service_Days', 'Prediction_Target' 
    ]
    # Filter columns that actually exist in the DataFrame before saving
    columns_to_save = [col for col in columns_to_save if col in df.columns]

    df[columns_to_save].to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n✅ Preprocessed Epic Games data saved to: {output_filename}")
    print(f"Total rows saved: {len(df)}")
else:
    print("DataFrame not loaded. Nothing to save.")

---
**Next Steps:**
1. Repeat this preprocessing for `Xbox.csv` and `PS.csv`.
2. Combine the three `_Preprocessed.csv` files.
3. Perform the detailed Feature Engineering for Publishers/Developers (Step 6).
4. Train a classification model (Step 7).
---